1. Mount Drive

Connect the notebook with my Drive, because files in Google Colab is temporary, any disconnection can cause loss. All the results csv files will be saved to our Drive automatically by mounting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Step 1: “strong-signal candidate collection” as a single, clean script.

This script:

searches only the compliance/regulation frameworks + a few high-precision intent terms,

fully paginates (no Top-N truncation),

saves consistent metadata + README text (for later gating and labeling),

is resume-safe (checkpointed),

and logs exactly why each repo was collected (the query that hit).

In [ ]:
from google.colab import files
files.upload()

{}

In [ ]:
!pip install -q --upgrade --no-deps huggingface_hub rapidfuzz tqdm pyyaml
import sys, pandas, huggingface_hub
print("Python:", sys.version)
print("pandas:", pandas.__version__)
print("huggingface_hub:", huggingface_hub.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 55.0 MB/s eta 0:00:00
Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
pandas: 2.2.2
huggingface_hub: 0.34.4


In [ ]:
import os
os.environ["HUGGING_FACE_HUB_TOKEN"] = "hf_xxxxxxxx" # Dear readers, please fill your own hugging token here - Wenjia 9/3/2025
print("Token set:", os.environ["HUGGING_FACE_HUB_TOKEN"][:6] + "..." + os.environ["HUGGING_FACE_HUB_TOKEN"][-4:])


Token set: hf_OtE...dFLz


Run Step 1:

In [ ]:
!python s1_candidatesCollector.py \
  --out /content/drive/MyDrive/candidates_step1_frameworks.csv --sleep 0.2 \
  --queries "GDPR" "HIPAA" "SOC 2" "ISO 27001" "PCI DSS" "SOX" "FedRAMP" "CMMC" \
            "NIST SP 800-53" "NIST SP 800-207" "NIST CSF" "CIS Controls" "CIS Controls V8" "CCPA" "CPRA"



[Init] huggingface_hub=0.34.4
/content/s1_candidatesCollector.py:70: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
[2025-09-03T14:04:59Z] Query: GDPR
[2025-09-03T14:05:33Z] Query done: GDPR (new rows: 149)
[2025-09-03T14:05:33Z] Query: HIPAA
[2025-09-03T14:05:36Z] Query done: HIPAA (new rows: 14)
[2025-09-03T14:05:36Z] Query: SOC 2
[2025-09-03T14:06:59Z] Query done: SOC 2 (new rows: 395)
[2025-09-03T14:06:59Z] Query: ISO 27001
[2025-09-03T14:07:00Z] Query done: ISO 27001 (new rows: 5)
[2025-09-03T14:07:00Z] Query: PCI DSS
[2025-09-03T14:07:00Z] Query done: PCI DSS (new rows: 0)
[2025-09-03T14:07:00Z] Query: SOX
[2025-09-03T14:07:02Z] Query done: SOX (new rows: 8)
[2025-09-03T14:07:02Z] Query: FedRAMP
[2025-09-03T14:07:03Z] Query done: FedRAMP (new rows: 1)
[2025-09

Repair missing README text

In [ ]:
!python s1b_repair_readmes.py \
  --in  "/content/drive/MyDrive/compliance-llm-taxonomy/candidates_step1_frameworks.csv" \
  --out "/content/drive/MyDrive/compliance-llm-taxonomy/candidates_step1_frameworks_repaired.csv" \
  --minlen 30 --sleep 0.01 --retries 2


README.md: 2.04kB [00:00, 3.21MB/s]
/content/s1b_repair_readmes.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '---
license: mit
widget:
- text: "We do not knowingly collect personal information from anyone under 16. We may limit how we collect, use and store some of the information of EU or EEA users between ages 13 and 16."
---

#### Example use:
```python
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained("PaDaS-Lab/gdpr-privacy-policy-ner", use_auth_token="AUTH_TOKEN")
model = AutoModelForTokenClassification.from_pretrained("PaDaS-Lab/gdpr-privacy-policy-ner", use_auth_token="AUTH_TOKEN")

ner = pipeline("ner", model=model, tokenizer=tokenizer)
example = "We do not knowingly collect personal information from anyone under 16. We may limit how we collect, use and store some of the information of EU or

Step 2 — gate & de-noise

In [ ]:
!python s2_gate_and_denoise.py \
  --in "/content/drive/MyDrive/compliance-llm-taxonomy/candidates_step1_frameworks_repaired.csv" \
  --outdir "/content/drive/MyDrive/compliance-llm-taxonomy" \
  --readme_min 120 --window 30

# Files written:
#   /content/drive/MyDrive/compliance-llm-taxonomy/step2_kept.csv
#   /content/drive/MyDrive/compliance-llm-taxonomy/step2_rejected.csv
#   /content/drive/MyDrive/compliance-llm-taxonomy/step2_family_map.csv


==== Step 2/3 Gate & De-noise Summary ====
Total candidates:            589
Kept before family collapse: 30
Kept (representatives only): 30
Rejected:                    559
Kept with CV/OCR flag:       0  (review if desired)
------------------------------------------
Top rejection reasons:
why_rejected
no_compliance_signal    258
placeholder_readme      168
empty_readme            133
------------------------------------------
Outputs written to: /content/drive/MyDrive/compliance-llm-taxonomy
 - step2_kept.csv
 - step2_rejected.csv
 - step2_family_map.csv


Step 3: Rescue base repos & Merge families

1.   scans readme_text in step2_kept/rejected CSVs for Hugging Face links,

2. fetches those base repos via the API,

3. runs the same gates as Step 2 (contentfulness + compliance relevance + noise),

4. merges the rescued bases with your kept set, and

5. outputs a clean, deduped set (one representative per family, preferring longer README).



In [ ]:
!python s3_rescue_base_repos.py \
  --kept "/content/drive/MyDrive/compliance-llm-taxonomy/step2_kept.csv" \
  --rejected "/content/drive/MyDrive/compliance-llm-taxonomy/step2_rejected.csv" \
  --outdir "/content/drive/MyDrive/compliance-llm-taxonomy" \
  --readme_min 120 --window 30

# Files written:
#   /content/drive/MyDrive/compliance-llm-taxonomy/step3_rescued_bases.csv
#   /content/drive/MyDrive/compliance-llm-taxonomy/step3_after_rescue.csv
#   /content/drive/MyDrive/compliance-llm-taxonomy/step3_rescue_log.csv


[Init] huggingface_hub=0.34.4
/content/s3_rescue_base_repos.py:85: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
[2025-09-03T17:12:50Z] Found 74 unique HF links; 45 not already in your set.
/content/s3_rescue_base_repos.py:223: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "collection_ts_utc": datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
==== Step 3 Rescue Summary ====
Candidates from links:    45
Rescued & kept:           0
Merged representatives:    30
--------------------------------
Outputs @ /content/drive/MyDrive/compliance-llm-taxonomy:
 - step3_rescued_bases.csv
 - step3_after_rescue.cs

**Step 3b. De-duplicates**

1. Collapse near-duplicates (GGUF/GPTQ/quant/“-v2/-beta” variants → pick one representative).

2. Generate a label sheet with taxonomy columns prefilled, so you can start annotating.




In [ ]:
!python s3b_dedup_and_label_prep.py \
  --in "/content/drive/MyDrive/compliance-llm-taxonomy/step3_after_rescue.csv" \
  --outdir "/content/drive/MyDrive/compliance-llm-taxonomy"


==== Dedup & Prefill Summary ====
Input rows:      30
Families:        20
Kept (unique):   20
Saved inventory: /content/drive/MyDrive/compliance-llm-taxonomy/step3_final_dedup.csv
Label sheet:     /content/drive/MyDrive/compliance-llm-taxonomy/step5_label_sheet.csv
